# 최근 4개 결산연도 보통주·우선주 배당

SEIBRO WebSquare의 배당내역상세에서 결산연도별 주당배당금, 시가배당률,
배당금과 배당성향을 확인합니다. 종목을 찾는 단계부터 웹만 사용합니다.

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/beaten-by-the-market/seibro-api/blob/main/scenarios/web/01_dividend_details.ipynb)

## 0. 준비
이 시나리오는 WebSquare 공개 화면을 조회하며 API 키가 필요 없습니다. 아래 두 셀을 실행해 패키지를 설치하고 실행 환경을 준비하세요.

In [1]:
# Colab에서만: 패키지 설치 (로컬은 이 레포를 pip install -e . 로 설치해 두면 됨)
import sys
if "google.colab" in sys.modules:
    !pip -q install "git+https://github.com/beaten-by-the-market/seibro-api.git" koreanize-matplotlib

In [2]:
SECRETS = []
# ── SEIBRO 공통 셀 (모든 시나리오 노트북 공통) ─────────────────────────────
# Colab: 왼쪽 🔑(Secrets)에 아래 SECRETS 이름으로 값을 넣고 "노트북 액세스"를 켜세요.
#        SEIBRO_API_KEY = 예탁결제원 SEIBRO Open API 인증키 (https://seibro.or.kr/OpenPlatform)
#        DART_API_KEY = OpenDART 인증키 (DART 데이터를 쓰는 노트북만)
#        KRX_ID / KRX_PW = KRX 정보데이터시스템 계정 (KRX 데이터를 쓰는 노트북만)
# 로컬: 환경변수 또는 작업 폴더(레포 루트)의 .env 를 읽습니다.
# 키·비밀번호 값은 어디에도 출력하지 않습니다. 있는지 여부만 보여 줍니다.
import os, sys, warnings
from datetime import date, datetime, timedelta

import pandas as pd

IN_COLAB = "google.colab" in sys.modules
warnings.filterwarnings("ignore", message="Unverified HTTPS request")

if IN_COLAB and SECRETS:
    from google.colab import userdata
    for _name in SECRETS:  # noqa: F821  (빌더가 이 셀 맨 위에 SECRETS 목록을 넣음)
        if os.environ.get(_name):
            continue
        try:
            _v = userdata.get(_name)
        except Exception:
            _v = None
        if not _v:
            from getpass import getpass
            _v = getpass(f"{_name}: ")
        os.environ[_name] = _v.strip()
elif SECRETS:
    from dotenv import find_dotenv, load_dotenv
    load_dotenv(find_dotenv(usecwd=True))

_missing = [
    n for n in SECRETS  # noqa: F821
    if not os.environ.get(n) and (IN_COLAB or not n.startswith("KRX_"))
]
if _missing:
    raise RuntimeError(f"{_missing} 가 없습니다. Colab Secrets 또는 .env 에 넣어 주세요.")

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")


def won(v, digits=1):
    """원 → '1.2조' / '5,027억' / '3,400만'."""
    if v is None or pd.isna(v):
        return "-"
    a = abs(v)
    if a >= 1e12:
        return f"{v / 1e12:.{digits}f}조"
    if a >= 1e8:
        return f"{v / 1e8:,.0f}억"
    if a >= 1e4:
        return f"{v / 1e4:,.0f}만"
    return f"{v:,.0f}"


def shares(v):
    """주식수 → '23,607,980주'."""
    return "-" if v is None or pd.isna(v) else f"{int(v):,}주"


# 한글 차트 폰트
import matplotlib.pyplot as plt
try:
    import koreanize_matplotlib  # noqa: F401  (Colab: 위 셀에서 설치)
except ImportError:
    if sys.platform.startswith("win"):
        plt.rcParams["font.family"] = "Malgun Gothic"
    elif sys.platform == "darwin":
        plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

def _status(n):
    if os.environ.get(n):
        return "있음"
    # 로컬에서 krx-data-api를 편집 설치(pip install -e)했다면 그 레포의 .env를 패키지가 직접 읽는다
    return "없음" if IN_COLAB or not n.startswith("KRX_") else "krx-data-api 자체 .env 사용"


credential_status = ", ".join(f"{n}: {_status(n)}" for n in SECRETS)
print("SEIBRO 준비 완료", "(Colab)" if IN_COLAB else "(로컬)",
      "| " + (credential_status or "웹 조회 · 키 필요 없음"))

SEIBRO 준비 완료 (로컬) | 웹 조회 · 키 필요 없음


In [3]:
from IPython.display import display

from seibro_api import get_dividend_details

STOCK_CODE = "005930"  # 삼성전자

In [4]:
data = get_dividend_details(STOCK_CODE, save_csv=False, use_openapi=False)
if data.empty:
    print("조회된 배당내역이 없습니다.")
else:
    display(data)


[Seibro] 배당내역상세 조회
  -> 삼성전자 (005930, 고객번호: 593)
  OK 보통주: 10개 항목


  OK 우선주: 10개 항목


,종목코드,회사명,주식종류,항목,2025,2024,2023,2022
0,005930,삼성전자,보통주,당기순이익(백만),45206805,34451351,15487100,55654077
1,005930,삼성전자,보통주,유보이익(백만),13937629,40265603,8047633,48948445
2,005930,삼성전자,보통주,배당금(백만),9762561,8621524,8620366,8620366
3,005930,삼성전자,보통주,배당금(현금)(백만),9762561,8621524,8620366,8620366
4,005930,삼성전자,보통주,배당금(주식)(백만),0,0,0,0
5,005930,삼성전자,보통주,배당성향(%),25.1,29.18,67.78,17.92
6,005930,삼성전자,보통주,DPS(원),1668,1446,1444,1444
7,005930,삼성전자,보통주,DPS(현금)(원),1668,1446,1444,1444
8,005930,삼성전자,보통주,DPS(주식)(원),0,0,0,0
9,005930,삼성전자,보통주,시가배당률(현금)(%),1.39,2.72,1.84,2.61


화면은 최근 4개 결산연도를 반환합니다. 권리기준일별 배당과 지급일은
[공식 API 배당 이력](../openapi/03_dividend_history.md)에서 조회할 수 있습니다.